# Mobilis Telecom - RAG Pipeline (standalone)
This notebook builds and tests the RAG pipeline **on its own**, independent of the intent classifier and the final LLM interface. Goal: prove retrieval quality first, before wiring anything else in.

**Pipeline:**
1. Load knowledge base (Bitext telco `instruction` + `response` pairs)
2. Chunk it
3. Embed chunks
4. Store in ChromaDB with metadata (`intent`, `category`)
5. Query: embed question -> retrieve top-k chunks -> (optionally) filter by intent
6. Sanity-check retrieval quality with sample questions
7. Assemble a final prompt for the LLM (generation itself is a stub here - swap in OpenAI/Ollama)

## 1. Install Dependencies

In [1]:
!pip install -q chromadb sentence-transformers pandas

# NOTE: if you hit "sentence_transformers python package is not installed" after this,
# it means transformers was already loaded in this session before the install finished.
# Fix: Runtime -> Restart session, then re-run this notebook from the top.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 123.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 91.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency 

## 2. Imports

In [ ]:
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
import uuid


## 3. Load the Knowledge Base
Using the same Bitext dataset. Each row becomes one chunk: the customer `instruction` paired with the official `response` — this is what gets retrieved and handed to the LLM as context.

In [ ]:
df = pd.read_csv("hf://datasets/bitext/Bitext-telco-llm-chatbot-training-dataset/bitext-telco-llm-chatbot-training-dataset.csv")
df = df.drop_duplicates().dropna()
print(df.shape)
df.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


(26000, 5)


,instruction,intent,category,tags,response
0,there are charges on my phone bill that i do n...,dispute_invoice,BILLING,BCELQZ,If you have noticed discrepancies in your bill...
1,"my internet bill is incorrect, can you help me...",dispute_invoice,BILLING,BCILZ,If there is an issue with your bill and you wo...
2,my invoice is not corect challenge it,dispute_invoice,BILLING,BCELQZ,If you find any discrepancies in your invoice ...
3,"I don't recognize some charges on my bill, can...",dispute_invoice,BILLING,BCIL,If you have found charges on your bill that yo...
4,i have to dispute a fucking bill i need help,dispute_invoice,BILLING,BCQW,If you believe there is an error on your bill ...


## 4. Chunking
Bitext responses are short (a paragraph at most), so we do **not** need to split them further — one row = one chunk. The chunk text combines the response with a bit of context (category/intent) so the embedding captures topic, not just surface wording.

If you later add longer real Mobilis documents (PDFs, policy pages), split those with a token-window chunker (e.g. 300 tokens, 50 overlap) - `langchain.text_splitter.RecursiveCharacterTextSplitter` or a simple manual splitter both work. Short FAQ-style data like this one doesn'''t need it.

In [ ]:
def build_chunk(row):
    """One chunk = the answer text, with light context prepended for retrieval quality."""
    return f"Category: {row['category']} | Intent: {row['intent']}\nQ: {row['instruction']}\nA: {row['response']}"

chunks = []
metadatas = []
ids = []

for _, row in df.iterrows():
    chunks.append(build_chunk(row))
    metadatas.append({"intent": row["intent"], "category": row["category"]})
    ids.append(str(uuid.uuid4()))

print(f"{len(chunks)} chunks built")
print(chunks[0])


26000 chunks built
Category: BILLING | Intent: dispute_invoice
Q: there are charges on my phone bill that i do not recogniae challenge it
A: If you have noticed discrepancies in your bill and wish to contest them, please adhere to the following steps:

1. Log in to your account on {{WEBSITE_URL}}.
2. Navigate to the {{INVOICE_SECTION}} section.
3. Select the bill you wish to dispute.
4. Click on the {{DISPUTE_INVOICE_OPTION}} to dispute the charge.
5. Fill in the required information and submit your dispute.

Our team will review your submission and get back to you within {{DAYS_NUMBER}} business days.


## 5. Embeddings
Using `sentence-transformers/all-MiniLM-L6-v2` — fast, good quality, 384-dim. If you expect French customer messages too, switch to `paraphrase-multilingual-MiniLM-L12-v2` (same interface, just a different model name below).

In [ ]:
EMBED_MODEL = "all-MiniLM-L6-v2"  # or "paraphrase-multilingual-MiniLM-L12-v2" for French/multilingual

try:
    embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
    # force it to actually load the model now, so any version-mismatch error surfaces here
    embedding_fn(["test"])
    print("Using sentence-transformers:", EMBED_MODEL)
except Exception as e:
    print(f"sentence-transformers embedding failed ({type(e).__name__}: {e}).")
    print("Falling back to ChromaDB's built-in embedding model instead.")
    embedding_fn = embedding_functions.DefaultEmbeddingFunction()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Using sentence-transformers: all-MiniLM-L6-v2


## 6. Store in ChromaDB
For a small/medium knowledge base like this, embed and insert in batches to avoid memory spikes. Chroma stores the vectors, the raw chunk text, and the metadata together — metadata is what lets us filter by intent later.

In [ ]:
client = chromadb.Client()  # in-memory for this notebook; use chromadb.PersistentClient(path="./chroma_db") to persist to disk

# Recreate collection fresh each run while iterating
try:
    client.delete_collection("mobilis_kb")
except Exception:
    pass

collection = client.create_collection(name="mobilis_kb", embedding_function=embedding_fn)

BATCH = 200
for i in range(0, len(chunks), BATCH):
    collection.add(
        documents=chunks[i:i+BATCH],
        metadatas=metadatas[i:i+BATCH],
        ids=ids[i:i+BATCH],
    )

print("Stored:", collection.count(), "chunks")


Stored: 26000 chunks


## 7. Retrieval Function
`intent_filter=None` searches the whole knowledge base (use this when the classifier returned `"unknown"` / low confidence). Pass an intent string to restrict the search to just that category once you trust the classifier's prediction.

In [2]:
def retrieve(query, k=3, intent_filter=None):
    where = {"intent": intent_filter} if intent_filter else None
    results = collection.query(
        query_texts=[query],
        n_results=k,
        where=where,
    )
    return list(zip(results["documents"][0], results["metadatas"][0], results["distances"][0]))


## 8. Sanity-Check Retrieval Quality
This is the step to actually look at before moving on. For each test question, read the retrieved chunks - do they genuinely answer the question? Distance (lower = more similar) gives a rough sense of match quality, but eyeballing the text matters more.

In [ ]:
test_queries = [
    "my internet is very slow today",
    "there are charges on my bill I do not recognize",
    "I want to dispute my invoice",
    "how do I schedule a payment",
]

for q in test_queries:
    print(f"QUERY: {q}")
    for doc, meta, dist in retrieve(q, k=3):
        print(f"  [{meta['intent']}] (distance={dist:.3f})")
        print(f"  {doc[:200]}")
        print()
    print("=" * 70)


QUERY: my internet is very slow today
  [report_poor_signal_coverage] (distance=0.576)
  Category: COMPLAINTS | Intent: report_poor_signal_coverage
Q: my internet is slow i need to complakn about poor signal coverage
A: We recognize the importance of reliable signal coverage for effective

  [report_poor_signal_coverage] (distance=0.606)
  Category: COMPLAINTS | Intent: report_poor_signal_coverage
Q: my internet is slow i need to report poor signal coverage
A: We recognize the importance of reliable signal coverage for effective communi

  [report_poor_signal_coverage] (distance=0.608)
  Category: COMPLAINTS | Intent: report_poor_signal_coverage
Q: my internet is slow id like to report poor signal coverage
A: We recognize the importance of reliable signal coverage for effective commun

QUERY: there are charges on my bill I do not recognize
  [dispute_invoice] (distance=0.346)
  Category: BILLING | Intent: dispute_invoice
Q: there are charges on my bill that I do not recognize, could yo

## 9. Filtered Retrieval (with intent, once classifier is wired in)
This is where the intent classifier from the other notebook plugs in: its predicted label becomes `intent_filter` below. If confidence was low (`"unknown"`), pass `intent_filter=None` and search broadly instead.

In [ ]:
# Example: pretend the classifier predicted "dispute_invoice" with high confidence
results = retrieve("there are charges I dont recognize", k=3, intent_filter="dispute_invoice")
for doc, meta, dist in results:
    print(f"[{meta['intent']}] (distance={dist:.3f})\n{doc}\n")


[dispute_invoice] (distance=0.482)
Category: BILLING | Intent: dispute_invoice
Q: there are charges on my invoice that i dont recognize can uchalenge it
A: If you have noticed unfamiliar charges on your invoice and wish to dispute them, please adhere to the following steps:

1. Log in to your account on {{WEBSITE_URL}}.
2. Navigate to the {{INVOICE_SECTION}} section.
3. Select the bill you wish to dispute.
4. Click on the {{DISPUTE_INVOICE_OPTION}} to dispute the charge.
5. Fill in the required information and submit your dispute.

Our team will review your submission and get back to you within {{DAYS_NUMBER}} business days.

[dispute_invoice] (distance=0.486)
Category: BILLING | Intent: dispute_invoice
Q: i dont recognize some charges on my internet bill can uchallenge it
A: If you have identified charges on your bill that you do not recognize and wish to challenge, please adhere to the following steps:

1. Log in to your account on {{WEBSITE_URL}}.
2. Navigate to the {{INVOICE_SECTIO

## 10. Prompt Assembly
Build the final prompt sent to the LLM: the retrieved chunks as grounding context, plus the user's actual question. The instruction to only use the provided context is what keeps the LLM from hallucinating outside your knowledge base.

In [ ]:
def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(doc for doc, meta, dist in retrieved_chunks)
    prompt = f"""You are a Mobilis customer support assistant. Answer the customers question using ONLY the context below.
If the context does not contain the answer, say you dont have that information and offer to escalate to a human agent.

Context:
{context}

Customer question: {question}

Answer in a natural, helpful, concise tone:"""
    return prompt

sample_q = "my internet is very slow today"
retrieved = retrieve(sample_q, k=3)
prompt = build_prompt(sample_q, retrieved)
print(prompt)


You are a Mobilis customer support assistant. Answer the customers question using ONLY the context below.
If the context does not contain the answer, say you dont have that information and offer to escalate to a human agent.

Context:
Category: COMPLAINTS | Intent: report_poor_signal_coverage
Q: my internet is slow i need to complakn about poor signal coverage
A: We recognize the importance of reliable signal coverage for effective communication. Please follow the steps below to report any issues you are experiencing with poor signal coverage:

1. Visit our website at {{WEBSITE_URL}}.
2. Navigate to the {{SUPPORT_SECTION}} section.
3. Select the option labeled {{REPORT_POOR_SIGNAL_COVERAGE_OPTION}}.
4. Fill in the required details, including your location and issue description.
5. Submit the form to notify our team.

Our team will assess your report and take appropriate measures to enhance network coverage in your vicinity.

Category: COMPLAINTS | Intent: report_poor_signal_coverage
Q:

## 11. Generation (stub — plug in OpenAI or Ollama)
Retrieval and prompt assembly are now validated independently. This cell is where generation plugs in — pick ONE of the two options below.

In [ ]:
# --- Option A: OpenAI / GPT ---
# from openai import OpenAI
# client_llm = OpenAI(api_key="YOUR_API_KEY")
#
# def generate_answer(prompt):
#     response = client_llm.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return response.choices[0].message.content

# --- Option B: Local Ollama (free, runs on your machine) ---
# !pip install -q ollama
# import ollama
#
# def generate_answer(prompt):
#     response = ollama.chat(model="mistral", messages=[{"role": "user", "content": prompt}])
#     return response["message"]["content"]

# Uncomment one option above, then:
# answer = generate_answer(prompt)
# print(answer)


## 12. Connect the Intent Classifier -> End-to-End Test
Loads the saved model from `01_intent_classifier.ipynb` (`mobilis_intent_model/`) and uses its predicted intent to filter retrieval. If confidence is below the threshold, the classifier returns `"unknown"` and we fall back to unfiltered (broad) retrieval instead of trusting a shaky label.

**Requires:** the `mobilis_intent_model/` folder (saved in section 13 of the classifier notebook) must be present in this notebook's working directory.

In [ ]:
from transformers import pipeline

CONFIDENCE_THRESHOLD = 0.5  # keep in sync with the classifier notebook

classifier = pipeline(
    "text-classification",
    model="mobilis_intent_model",
    tokenizer="mobilis_intent_model"
)

def predict_intent(message, threshold=CONFIDENCE_THRESHOLD):
    prediction = classifier(message.lower())[0]
    intent = prediction["label"]
    score = prediction["score"]
    if score < threshold:
        return {"intent": "unknown", "raw_intent": intent, "confidence": score}
    return {"intent": intent, "raw_intent": intent, "confidence": score}


def answer_question(question, k=3, verbose=True):
    """Full pipeline: classify intent -> filter retrieval -> assemble prompt.
    Generation is still a stub (see section 11) until an LLM is wired in."""
    intent_result = predict_intent(question)
    intent_filter = None if intent_result["intent"] == "unknown" else intent_result["intent"]

    retrieved = retrieve(question, k=k, intent_filter=intent_filter)
    prompt = build_prompt(question, retrieved)

    if verbose:
        print(f"Predicted intent : {intent_result[\'intent\']} "
              f"(raw={intent_result[\'raw_intent\']}, confidence={intent_result[\'confidence\']:.2f})")
        print(f"Retrieval filter : {intent_filter}")
        print("-" * 70)

    # answer = generate_answer(prompt)   # uncomment once Section 11 generation is wired in
    # return answer
    return prompt  # placeholder: returns the assembled prompt until generation is wired in


# Quick test across a few messages, including one that should trigger "unknown"
test_msgs = [
    "my internet is very slow today",
    "there are charges on my bill I do not recognize",
    "asdkj qweoiqwe random gibberish message",
]

for msg in test_msgs:
    print("=" * 70)
    print("QUESTION:", msg)
    print(answer_question(msg))
    print()
